# Medical QA Capstone — Final Analysis (90 questions)

**Project:** Evaluating Multi-Agent Reasoning Strategies for Complex Medical Question Answering  
**Benchmark subset:** MedicalAgentsBench / MedQA `test_hard`, Q11–Q100  
**Inputs:** The five completed experiment CSVs in the same folder as this notebook.  
**Outputs:** `analysis_outputs/` containing validated tables, tests, publication-style figures, and a Markdown findings report.

**Instructions:** Place this notebook beside the CSVs. In VS Code select your `.venv` Python kernel and click **Run All**. The notebook **does not call Groq or Hugging Face**. It never modifies the original experiment CSVs.  

**Interpretation rules:** Self-Consistency uses majority vote, and a 3-way tie is broken using sample 1. Critic-guided cost includes its baseline + critic call. Each *n*-agent configuration includes n independent expert calls + **judge_n**, not the judges from smaller configurations. Logged API latencies do not include deliberate delays, rate-limit waits, or overhead. Single-letter outputs do not measure explanation quality.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from IPython.display import display

ROOT = Path.cwd()
OUT = ROOT / "analysis_outputs"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
EXPECTED_Q = list(range(11, 101))
N = len(EXPECTED_Q)

FILES = {
    "baseline": "formal_baseline_results.csv",
    "structured": "structured_reasoning_results.csv",
    "critic": "critic_results.csv",
    "self_consistency": "self_consistency_samples.csv",
    "multi_agent": "multi_agent_results.csv",
}

def locate(filename):
    p = ROOT / filename
    if p.is_file():
        return p
    alt = sorted(ROOT.glob(f"{Path(filename).stem}(*){Path(filename).suffix}"))
    if len(alt) == 1:
        print(f"Using {alt[0].name} instead of {filename}")
        return alt[0]
    raise FileNotFoundError(f"Could not uniquely locate {filename} in {ROOT}. Place the notebook in the folder containing your five CSV files.")

data = {name: pd.read_csv(locate(filename)) for name, filename in FILES.items()}
print("Working folder:", ROOT)
print("File shapes:", {k: v.shape for k,v in data.items()})

## A. Validate the saved experiments

This checks unique records, exact question coverage, answer-key and benchmark-ID alignment, correctness flags, and token accounting. It stops with an error rather than silently analysing corrupt or incomplete files.

In [ ]:
baseline = data["baseline"].copy()
structured = data["structured"].copy()
critic = data["critic"].copy()
sc = data["self_consistency"].copy()
ma = data["multi_agent"].copy()

for name, d in data.items():
    if d.isna().any().any():
        raise ValueError(f"{name}: contains missing values in saved experimental records")
    if "Question_Number" not in d:
        raise ValueError(f"{name}: missing Question_Number")
    if sorted(d.Question_Number.unique().tolist()) != EXPECTED_Q:
        raise ValueError(f"{name}: expected Q11 through Q100 exactly")

for name in ("baseline", "structured", "critic"):
    d = data[name]
    if len(d) != N or d.Question_Number.duplicated().any():
        raise ValueError(f"{name}: expected one record per question, 90 total")

if len(sc) != N*3 or sc.duplicated(["Question_Number", "Sample_Number"]).any():
    raise ValueError("self_consistency: missing or duplicated sample rows")
if not (sc.groupby("Question_Number").Sample_Number.apply(lambda x: set(x)=={1,2,3})).all():
    raise ValueError("self_consistency: each question needs sample numbers 1, 2, and 3")

stages = {"expert_1", "judge_1", "expert_2", "judge_2", "expert_3", "judge_3"}
if len(ma) != N*6 or ma.duplicated(["Question_Number", "Stage"]).any():
    raise ValueError("multi_agent: missing or duplicated stage rows")
if not (ma.groupby("Question_Number").Stage.apply(lambda x: set(x)==stages)).all():
    raise ValueError("multi_agent: missing expert/judge stages for one or more questions")
for i in (1,2,3):
    if not (ma.loc[ma.Stage.isin([f"expert_{i}",f"judge_{i}"]), "Agent_Count"]==i).all():
        raise ValueError(f"multi_agent: unexpected Agent_Count for stage {i}")

base_key = baseline.set_index("Question_Number").Correct.astype(str).str.upper()
base_id = baseline.set_index("Question_Number").Benchmark_ID.astype(str)
for name, d in data.items():
    d2 = d.set_index("Question_Number")
    if not (d2.Correct.astype(str).str.upper() == base_key.reindex(d2.index)).all():
        raise ValueError(f"{name}: answer key differs from baseline")
    if "Benchmark_ID" in d:
        if not (d2.Benchmark_ID.astype(str) == base_id.reindex(d2.index)).all():
            raise ValueError(f"{name}: benchmark ID differs from baseline")

for name, d in data.items():
    if not np.allclose(d.Input_Tokens + d.Output_Tokens, d.Total_Tokens):
        raise ValueError(f"{name}: token totals do not add up")
    for col in ["Response_Time", "Input_Tokens", "Output_Tokens", "Total_Tokens"]:
        if (d[col] < 0).any():
            raise ValueError(f"{name}: negative value in {col}")
    pred_col = "Critic_Prediction" if name == "critic" else "Prediction"
    flag_col = "Critic_Correct" if name == "critic" else "Is_Correct"
    checked = d[pred_col].astype(str).str.strip().str.upper().eq(d.Correct.astype(str).str.upper())
    observed = d[flag_col].astype(str).str.lower().eq("true")
    if not (checked == observed).all():
        raise ValueError(f"{name}: correctness flag disagrees with saved prediction")

bpred = baseline.set_index("Question_Number").Prediction.astype(str).str.upper()
if not (critic.set_index("Question_Number").Baseline_Prediction.astype(str).str.upper() == bpred).all():
    raise ValueError("critic: baseline predictions do not match baseline file")
if not (critic.Baseline_Correct.astype(str).str.lower().eq("true").values ==
        critic.Baseline_Prediction.astype(str).str.upper().eq(critic.Correct.astype(str).str.upper()).values).all():
    raise ValueError("critic: Baseline_Correct fields do not match")

print(f"PASS: {N} aligned questions; {sum(len(d) for d in data.values())} total records; no duplicates or mismatched correctness flags.")

## B. Consolidate predictions, agreement and the per-configuration costs

The output is one row per question. Cost measures are calculated from the actual API-stage records. A three-way Self-Consistency tie uses the first sampled answer as an explicit, reproducible rule.

In [ ]:
wide = pd.DataFrame(index=EXPECTED_Q)
wide.index.name = "Question_Number"
wide["Benchmark_ID"] = baseline.set_index("Question_Number").Benchmark_ID
wide["Correct"] = base_key
methods = ["Baseline", "Structured", "Critic", "SelfConsistency", "Judge1", "Judge2", "Judge3"]

for method, d, pred_col in [
    ("Baseline",baseline,"Prediction"),
    ("Structured",structured,"Prediction"),
    ("Critic",critic,"Critic_Prediction"),
]:
    wide[f"{method}_Pred"] = d.set_index("Question_Number")[pred_col].astype(str).str.upper()

samples = sc.pivot(index="Question_Number",columns="Sample_Number",values="Prediction").loc[EXPECTED_Q]
samples = samples.astype(str).apply(lambda s:s.str.upper())
for num in (1,2,3):
    wide[f"Sample{num}_Pred"] = samples[num]

def vote(row):
    counts = row.value_counts()
    return counts.index[0] if counts.iloc[0]>=2 else row.iloc[0]
wide["SelfConsistency_Pred"] = samples.apply(vote,axis=1)
wide["SelfConsistency_Agreement"] = samples.apply(
    lambda r: "Unanimous" if r.nunique()==1 else ("2-to-1 majority" if r.nunique()==2 else "3-way tie"), axis=1)
for n in (1,2,3):
    wide[f"Judge{n}_Pred"] = ma.loc[ma.Stage==f"judge_{n}"].set_index("Question_Number").Prediction.astype(str).str.upper()

for method in methods:
    wide[f"{method}_Acc"] = (wide[f"{method}_Pred"]==wide["Correct"]).astype(int)
wide["Sample1_Acc"] = (wide["Sample1_Pred"]==wide["Correct"]).astype(int)

num_cols = ["Input_Tokens", "Output_Tokens", "Total_Tokens", "Response_Time"]
for method, d in [("Baseline",baseline), ("Structured",structured)]:
    for col in num_cols:
        wide[f"{method}_{col}"] = d.set_index("Question_Number")[col]
for col in num_cols:
    wide[f"Critic_{col}"] = (baseline.set_index("Question_Number")[col]
                                + critic.set_index("Question_Number")[col])
    wide[f"SelfConsistency_{col}"] = sc.groupby("Question_Number")[col].sum().reindex(EXPECTED_Q)
    for n in (1,2,3):
        required_stages = [f"expert_{i}" for i in range(1,n+1)] + [f"judge_{n}"]
        wide[f"Judge{n}_{col}"] = ma[ma.Stage.isin(required_stages)].groupby("Question_Number")[col].sum().reindex(EXPECTED_Q)

for n in (1,2,3):
    wide[f"Judge{n}_Calls"] = n+1
wide["Baseline_Calls"] = 1
wide["Structured_Calls"] = 1
wide["Critic_Calls"] = 2
wide["SelfConsistency_Calls"] = 3
wide.to_csv(OUT / "question_level_results.csv")

agreement = wide.SelfConsistency_Agreement.value_counts().reindex(
    ["Unanimous", "2-to-1 majority", "3-way tie"], fill_value=0)
print("Self-Consistency agreement:")
display(agreement.to_frame("Questions"))
print("Sample 1 accuracy:", int(wide.Sample1_Acc.sum()), "/",N)
print("Self-Consistency voting accuracy:", int(wide.SelfConsistency_Acc.sum()), "/",N)
print("Saved:",OUT / "question_level_results.csv")

## C. Accuracy, 95% confidence intervals and paired tests

Wilson intervals summarize accuracy uncertainty. Exact paired McNemar tests compare methods on the **same 90 questions**. Six comparisons against baseline are Holm-adjusted; agent-count contrasts are labelled exploratory and Holm-adjusted within their own three-comparison family.

In [ ]:
display_names = {
    "Baseline":"Direct baseline", "Structured":"Structured prompting",
    "Critic":"Critic-guided refinement", "SelfConsistency":"Self-Consistency",
    "Judge1":"1 agent + judge", "Judge2":"2 agents + judge", "Judge3":"3 agents + judge"
}

summary_rows=[]
for m in methods:
    correct = int(wide[f"{m}_Acc"].sum())
    low, high = proportion_confint(correct, N, alpha=0.05, method="wilson")
    summary_rows.append({
        "Method":display_names[m], "Code":m, "N":N, "Correct":correct,
        "Accuracy":correct/N, "CI_Lower":low,"CI_Upper":high,
        "Gain_vs_Baseline_pp":100*(correct-int(wide.Baseline_Acc.sum()))/N,
        "Calls_per_Question":int(wide[f"{m}_Calls"].iloc[0]),
        "Mean_Input_Tokens":wide[f"{m}_Input_Tokens"].mean(),
        "Mean_Output_Tokens":wide[f"{m}_Output_Tokens"].mean(),
        "Mean_Total_Tokens":wide[f"{m}_Total_Tokens"].mean(),
        "Mean_API_Latency_s":wide[f"{m}_Response_Time"].mean(),
        "Tokens_per_Correct_Answer":wide[f"{m}_Total_Tokens"].sum()/correct if correct else np.nan,
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT/"method_summary.csv",index=False)
display(summary[["Method","Correct","N","Accuracy","CI_Lower","CI_Upper","Mean_Total_Tokens","Mean_API_Latency_s"]].round(3))

def mcnemar_exact(m1,m2):
    a=wide[f"{m1}_Acc"].astype(bool)
    b=wide[f"{m2}_Acc"].astype(bool)
    improved=int((~a & b).sum())
    worsened=int((a & ~b).sum())
    n=improved+worsened
    p=binomtest(min(improved,worsened), n, p=0.5,alternative="two-sided").pvalue if n else 1.0
    return {"Reference":m1,"Comparison":m2,"Wrong_to_Correct":improved,
            "Correct_to_Wrong":worsened,"Net_Correct_Change":improved-worsened,"P_Exact":p}

vs_base=pd.DataFrame([mcnemar_exact("Baseline",m) for m in methods if m!="Baseline"])
vs_base["P_Holm_6"] = multipletests(vs_base.P_Exact,method="holm")[1]
vs_base["Family"] = "Six comparisons vs baseline"
agent_tests=pd.DataFrame([mcnemar_exact(a,b) for a,b in [
    ("Judge1","Judge2"),("Judge2","Judge3"),("Judge1","Judge3")]])
agent_tests["P_Holm_3"] = multipletests(agent_tests.P_Exact,method="holm")[1]
agent_tests["Family"] = "Three exploratory agent-count contrasts"
tests=pd.concat([vs_base,agent_tests],ignore_index=True)
tests.to_csv(OUT/"paired_mcnemar_tests.csv",index=False)
print("Paired comparisons versus baseline (Holm correction):")
display(vs_base.round(4))
print("Exploratory agent-count comparisons:")
display(agent_tests.round(4))

# For every method, preserve concrete question IDs that were corrected or worsened.
flips=[]
for m in methods[1:]:
    for q,row in wide.iterrows():
        if row.Baseline_Acc != row[f"{m}_Acc"]:
            flips.append({"Question_Number":q, "Method":display_names[m],
                          "Direction":"Corrected baseline error" if row[f"{m}_Acc"] else "Changed correct baseline answer to wrong",
                          "Baseline_Pred":row.Baseline_Pred,"Method_Pred":row[f"{m}_Pred"], "Correct":row.Correct})
pd.DataFrame(flips).to_csv(OUT/"changed_questions.csv",index=False)
print("Saved summary, paired tests, and changed-question IDs.")

## D. Publication-style figures

Figures are saved as high-resolution PNG files. These illustrate observed effects; statistical significance is evaluated from the paired tests above.

In [ ]:
plt.rcParams.update({"figure.dpi":130, "savefig.dpi":220, "font.size":10})

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / name, bbox_inches="tight")
    plt.show()
    plt.close()

# Figure 1 — accuracy with Wilson 95% confidence intervals
fig,ax=plt.subplots(figsize=(10,5.1))
x=np.arange(len(summary))
y=summary.Accuracy.to_numpy()*100
low_err=(summary.Accuracy-summary.CI_Lower).to_numpy()*100
high_err=(summary.CI_Upper-summary.Accuracy).to_numpy()*100
ax.bar(x,y,color="steelblue",alpha=.85)
ax.errorbar(x,y,yerr=[low_err,high_err],fmt="none",ecolor="black",capsize=4)
ax.set_xticks(x,summary.Method,rotation=24,ha="right")
ax.set_ylim(0,max(70,float((summary.CI_Upper*100).max()+6)))
ax.set_ylabel("Accuracy (%)")
ax.set_title("Medical QA accuracy (Q11–Q100, n=90): 95% Wilson intervals")
for i,(v,n) in enumerate(zip(y,summary.Correct)):
    ax.text(i,v+1.1,f"{n}/90",ha="center",fontsize=9)
savefig("01_accuracy_with_CI.png")

# Figure 2 — baseline corrections and regressions
fig,ax=plt.subplots(figsize=(9.5,4.9))
z=vs_base.set_index("Comparison").reindex(methods[1:])
ypos=np.arange(len(z))
ax.barh(ypos,z.Wrong_to_Correct.values,color="seagreen",label="Baseline wrong → method correct")
ax.barh(ypos,-z.Correct_to_Wrong.values,color="indianred",label="Baseline correct → method wrong")
ax.axvline(0,color="black",linewidth=.8)
ax.set_yticks(ypos,[display_names[m] for m in z.index])
ax.invert_yaxis()
ax.set_xlabel("Number of changed question outcomes (90 paired questions)")
ax.set_title("Corrections versus regressions relative to direct baseline")
ax.legend(loc="lower right",fontsize=8)
savefig("02_corrected_and_worsened.png")

# Figure 3 — number of agents, accuracy and tokens
agents=summary[summary.Code.isin(["Judge1","Judge2","Judge3"])].copy()
counts=np.array([1,2,3])
fig,ax1=plt.subplots(figsize=(7,4.8))
ax1.plot(counts,agents.Accuracy*100,"o-",linewidth=2,label="Accuracy (%)",color="royalblue")
ax1.set_xlim(.85,3.15)
ax1.set_xticks(counts)
ax1.set_ylim(0,75)
ax1.set_xlabel("Number of independent specialists (plus one judge)")
ax1.set_ylabel("Accuracy (%)",color="royalblue")
ax2=ax1.twinx()
ax2.plot(counts,agents.Mean_Total_Tokens,"s--",linewidth=2,label="Tokens/question",color="darkorange")
ax2.set_ylabel("Average total tokens/question",color="darkorange")
ax1.set_title("Observed agent-count trade-off (exploratory)")
savefig("03_agent_count_and_cost.png")

# Figure 4 — accuracy versus tokens
fig,ax=plt.subplots(figsize=(8.7,5.1))
for _,r in summary.iterrows():
    ax.scatter(r.Mean_Total_Tokens,r.Accuracy*100,s=55)
    ax.annotate(r.Method,(r.Mean_Total_Tokens,r.Accuracy*100),xytext=(5,4),textcoords="offset points",fontsize=8)
ax.set_xlabel("Average total tokens per question")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy versus model-computation proxy")
ax.grid(alpha=.25)
savefig("04_accuracy_vs_tokens.png")

# Figure 5 — answer agreement distribution
fig,ax=plt.subplots(figsize=(6.8,4.3))
ax.bar(agreement.index,agreement.values,color=["teal","slateblue","darkorange"])
for i,v in enumerate(agreement.values):
    ax.text(i,v+1,str(v),ha="center")
ax.set_ylim(0,max(agreement.values)+10)
ax.set_ylabel("Number of questions")
ax.set_title("Self-Consistency agreement across three samples")
savefig("05_self_consistency_agreement.png")
print("Figures saved to:",FIG)

## E. Medical question categories (Objective 7)

Question categories **cannot be determined from answer letters or question IDs**. If you have an approved `question_categories.csv` with columns `Question_Number, Category` in the project folder, this section produces category-specific accuracy and a heatmap. Otherwise, it creates `analysis_outputs/question_categories_TO_FILL.csv` and skips the analysis safely. Assign labels using your medical question texts and a consistent codebook, not model performance. Interpret small categories as exploratory.

In [ ]:
category_path=ROOT/"question_categories.csv"
category_results=None
if not category_path.exists():
    template=wide.reset_index()[["Question_Number","Benchmark_ID"]].copy()
    template["Category"]=""
    template["Notes"]=""
    target=OUT/"question_categories_TO_FILL.csv"
    if not target.exists():
        template.to_csv(target,index=False)
    print("Category analysis skipped: question_categories.csv not provided.")
    print("Optional template created at:",target)
    print("Fill categories using actual question text, then save as question_categories.csv in the project folder and rerun.")
else:
    categories=pd.read_csv(category_path)
    needed={"Question_Number","Category"}
    if not needed.issubset(categories.columns):
        raise ValueError("question_categories.csv needs columns Question_Number and Category")
    if categories.Question_Number.duplicated().any():
        raise ValueError("question_categories.csv includes duplicate Question_Number values")
    categories=categories[categories.Question_Number.isin(EXPECTED_Q)].copy()
    categories["Category"]=categories["Category"].fillna("").astype(str).str.strip()
    if (categories.Category=="").any() or len(categories)!=N:
        print("Category file is incomplete: analysing only labeled questions, with an explicit sample-size warning.")
    valid=categories[categories.Category!=""]
    joined=wide.reset_index().merge(valid[["Question_Number","Category"]],on="Question_Number",how="inner")
    if not len(joined):
        print("No labeled questions yet; category analysis skipped.")
    else:
        rows=[]
        for cat,g in joined.groupby("Category"):
            for m in methods:
                rows.append({"Category":cat,"Method":display_names[m],"N":len(g),
                             "Correct":int(g[f"{m}_Acc"].sum()),"Accuracy":g[f"{m}_Acc"].mean()})
        category_results=pd.DataFrame(rows)
        category_results.to_csv(OUT/"medical_category_accuracy.csv",index=False)
        matrix=category_results.pivot(index="Category",columns="Method",values="Accuracy")
        matrix=matrix.reindex(columns=[display_names[m] for m in methods])
        ns=joined.groupby("Category").size()
        fig,ax=plt.subplots(figsize=(10,max(3.5,.5*len(matrix)+1.7)))
        im=ax.imshow(matrix.to_numpy()*100,aspect="auto",vmin=0,vmax=100,cmap="Blues")
        ax.set_xticks(range(len(matrix.columns)),matrix.columns,rotation=30,ha="right",fontsize=8)
        ax.set_yticks(range(len(matrix)),[f"{c} (n={ns[c]})" for c in matrix.index])
        ax.set_title("Exploratory accuracy by medical question category (%)")
        fig.colorbar(im,ax=ax,label="Accuracy (%)")
        savefig("06_medical_category_accuracy.png")
        display(category_results.round(3))
        print("Category comparisons are descriptive, especially for small n.")

## F. Automatically save a report and research notes

The report records validated results, tests, observed limitations, and future interpretation needs. It **does not** claim to have verified original question texts, option validity, prompt versions, model settings or category assignments, which require separate source/script checks.

In [ ]:
def pc(x): return f"{100*x:.1f}%"

def md_table(df,cols,formatters=None):
    formatters=formatters or {}
    x=df[cols].copy()
    for col,fn in formatters.items():
        x[col]=x[col].map(fn)
    return x.to_markdown(index=False)

report=[]
report.append("# MSc Medical QA Capstone — Reproducible Analysis Summary\n")
report.append(f"**Dataset:** MedicalAgentsBench / MedQA `test_hard`, Q11–Q100 (n={N}).\n")
report.append("**Data integrity:** Five files passed question coverage, duplicate, answer-key, ID, correctness-flag and token-total checks.\n")
report.append("## Accuracy and computational cost\n")
report.append(md_table(summary,["Method","Correct","Accuracy","CI_Lower","CI_Upper","Calls_per_Question","Mean_Total_Tokens","Mean_API_Latency_s"],{
    "Accuracy":pc,"CI_Lower":pc,"CI_Upper":pc,
    "Mean_Total_Tokens":lambda x:f"{x:,.1f}","Mean_API_Latency_s":lambda x:f"{x:.2f}"})+"\n")
report.append("## Paired comparisons with baseline\n")
report.append(md_table(vs_base,["Comparison","Wrong_to_Correct","Correct_to_Wrong","P_Exact","P_Holm_6"],{
    "P_Exact":lambda x:f"{x:.4f}","P_Holm_6":lambda x:f"{x:.4f}"})+"\n")
report.append("Six paired baseline comparisons were corrected using Holm's method; do not interpret unadjusted p-values as confirmatory results.\n")
report.append("## Agent-count contrasts (exploratory)\n")
report.append(md_table(agent_tests,["Reference","Comparison","Wrong_to_Correct","Correct_to_Wrong","P_Exact","P_Holm_3"],{
    "P_Exact":lambda x:f"{x:.4f}","P_Holm_3":lambda x:f"{x:.4f}"})+"\n")
report.append("Roles and candidate contexts change with agent count; these differences do not isolate a pure agent-count causal effect.\n")
report.append("## Self-Consistency\n")
report.append("\n".join([f"- {label}: {int(value)}/{N}" for label,value in agreement.items()])+"\n")
report.append(f"Sample 1: **{int(wide.Sample1_Acc.sum())}/{N}**. Vote (first sample breaks three-way ties): **{int(wide.SelfConsistency_Acc.sum())}/{N}**.\n")
report.append("## Medical categories\n")
report.append("Category-specific results saved as `medical_category_accuracy.csv`. Treat small subgroups as exploratory.\n" if category_results is not None else
              "Not evaluated yet: an externally reviewed medical question-category file is required; a blank labeling template has been generated.\n")
report.append("## Methodological boundaries\n")
report.append("- This analysis uses stored responses, not the full 862-question benchmark; results generalize only cautiously beyond these 90 MedQA questions.\n")
report.append("- Structured prompting requested deliberate consideration but did not necessarily reveal full Chain-of-Thought; judge/agent outputs are only answer letters.\n")
report.append("- Token counts and response latencies are proxies for computational cost, not monetary cost or actual end-to-end wall time. Rate-limit waits and manual sleep intervals are excluded.\n")
report.append("- Critic method cost includes its original baseline call. N-agent configurations each include n experts and only the corresponding judge; 1-, 2-, and 3-agent experiments share prior expert outputs.\n")
report.append("- Original benchmark question text/options and prompt/script provenance are not independently verified by CSV-only analysis. Invalid option letters require verification against actual option sets.\n")
report.append("- Paired tests use the same questions; conclusions are exploratory with a 90-question sample and pre-existing, non-preregistered comparisons.\n")
report.append("- No result here establishes clinical safety, real-world diagnostic reliability, or causal effects of agent count.\n")
report.append("\n## Research-output files\n")
report.append("- `question_level_results.csv`, `method_summary.csv`, `paired_mcnemar_tests.csv`, `changed_questions.csv`, and `figures/*.png`.\n")
report.append("- If supplied, `medical_category_accuracy.csv` and its heatmap.\n")
(OUT/"analysis_report.md").write_text("\n".join(report),encoding="utf-8")
print("COMPLETE — no LLM calls were made.")
print("Report:",OUT/"analysis_report.md")
print("Figures:",FIG)
print("Question-level results:",OUT/"question_level_results.csv")
print("Next: review report, source-question/prompt validation, medical categories, and case-based error analysis.")